# Cash Flow Generation

## Introduction

After generating the payment schedule, the next step in the pricing pipeline is to convert each payment period into a contractual cash flow.

Each cash flow represents a future coupon payment that will later be discounted to its present value using the yield curve.

This module generates the fixed-leg cash flows of a vanilla Interest Rate Swap.

## Objectives

At the end of this notebook you will understand:

- What a contractual cash flow is
- How coupon amounts are calculated
- How payment periods become cash flows
- The architecture of the Cash Flow Generator
- How to generate cash flows programmatically

## Position in the Pricing Engine

Market Quotes

↓

Yield Curve

↓

Trade Representation

↓

Schedule Generation

↓

Cash Flow Generation ← Current Module

↓

Discounting

↓

Present Value

↓

Risk Measures

## Financial Concepts

A cash flow is a future payment specified by the contract.

For a fixed-rate interest rate swap, each coupon payment is calculated as:

Coupon = Notional × Fixed Rate × Year Fraction

Example

Notional = $1,000,000

Fixed Rate = 5%

Year Fraction = 1.0

Coupon = $50,000

## Software Architecture

This module consists of two components.

CashFlow

Represents a single contractual payment.

CashFlowGenerator

Converts payment periods into contractual cash flows.

Keeping the domain object separate from the generator makes the design modular and easier to extend.

## Design Principles

This implementation follows several design principles.

- Immutable data objects
- Single Responsibility Principle
- Separation of domain objects and services
- Type safety
- Unit-testable implementation

## Implementation Overview

The CashFlowGenerator performs the following steps.

1. Iterate through each payment period.

2. Calculate the accrual year fraction.

3. Compute the coupon amount.

4. Construct a CashFlow object.

5. Return all contractual cash flows.

## Algorithm

For each payment period

↓

Compute Year Fraction

↓

Coupon = Notional × Rate × Year Fraction

↓

Create CashFlow

↓

Append to collection

↓

Return all cash flows

## Simplifying Assumptions

To keep the implementation focused, the following assumptions are made.

- Fixed-leg cash flows only
- ACT/365 day count
- Annual payments
- Single currency
- No principal exchange
- No amortization
- No business-day adjustments

In [9]:
from datetime import date

import pandas as pd

from src.cashflows.cashflow_generator import CashFlowGenerator
from src.common.enums import (
    DayCountConvention,
    FloatingIndex,
    PaymentFrequency,
    PayReceive,
)
from src.products.fixed_leg import FixedLeg
from src.products.floating_leg import FloatingLeg
from src.products.interest_rate_swap import InterestRateSwap
from src.products.trade import Trade
from src.schedule.schedule_generator import ScheduleGenerator

In [10]:
trade = Trade(
    trade_id="IRS001",
    counterparty="ABC Bank",
    currency="USD",
    effective_date=date(2026, 1, 1),
    maturity_date=date(2031, 1, 1),
)

fixed_leg = FixedLeg(
    notional=1_000_000,
    fixed_rate=0.05,
    payment_frequency=PaymentFrequency.ANNUAL,
    day_count=DayCountConvention.ACT_365,
    pay_receive=PayReceive.RECEIVE,
)

floating_leg = FloatingLeg(
    notional=1_000_000,
    payment_frequency=PaymentFrequency.ANNUAL,
    day_count=DayCountConvention.ACT_365,
    pay_receive=PayReceive.PAY,
    floating_index=FloatingIndex.SOFR,
    spread=0.0,
)

swap = InterestRateSwap(
    trade=trade,
    fixed_leg=fixed_leg,
    floating_leg=floating_leg,
)

In [11]:
schedule = ScheduleGenerator().generate(swap)

cashflows = CashFlowGenerator().generate(
    swap,
    schedule,
)

In [12]:
pd.DataFrame(
    [
        {
            "Payment Date": cf.payment_date,
            "Rate": cf.rate,
            "Year Fraction": cf.year_fraction,
            "Amount": cf.amount,
            "Type": cf.cashflow_type.value,
        }
        for cf in cashflows
    ]
)

,Payment Date,Rate,Year Fraction,Amount,Type
0,2027-01-01,0.05,1.00000,50000.000000,Fixed
1,2028-01-01,0.05,1.00000,50000.000000,Fixed
2,2029-01-01,0.05,1.00274,50136.986301,Fixed
3,2030-01-01,0.05,1.00000,50000.000000,Fixed
4,2031-01-01,0.05,1.00000,50000.000000,Fixed


In [13]:
assert len(cashflows) == 5

assert cashflows[0].amount == 50_000

print("✓ Module 6 completed successfully.")

✓ Module 6 completed successfully.


## Conclusion

The pricing engine now produces contractual cash flows from an Interest Rate Swap.

These cash flows become the inputs to the next stage of the pricing pipeline, where each payment will be discounted using the yield curve to calculate present value.

## Key Takeaways

- Payment schedules define when cash flows occur.
- Cash flows define how much will be paid.
- Coupon amounts depend on notional, rate, and accrual period.
- CashFlow objects are immutable.
- CashFlowGenerator converts schedules into contractual payments.
- The next module introduces discounting to calculate present value.